In [3]:
!pip install -q google-genai
import os
import json
from google import genai
from google.genai import types

# --- 1. SETUP & CONFIGURATION ---
GOOGLE_API_KEY = os.environ.get("GEMINI_API_KEY", "AQ.****************************")
client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL_ID = "gemini-3.1-flash-lite" 


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
# --- 2. KITCHEN STATE MANAGEMENT ---
class KitchenManager:
    def __init__(self, initial_inventory: list[str]):
        self.base_inventory = set(initial_inventory)
        self.inventory = set(initial_inventory)
        self.served_dish = None

    def reset(self):
        self.inventory = self.base_inventory.copy()
        self.served_dish = None
        
    def check_missing(self, required_items: list[str]) -> list[str]:
        return [item for item in required_items if item not in self.inventory]

kitchen = KitchenManager([
    "bread", "potato", "cheese", "lettuce", 
    "tomato", "onion", "butter", "oil", "salt", 
    "pasta", "pizza dough", "pizza sauce"
])

In [5]:
# --- 3. KITCHEN TOOLS WITH REAL-TIME TELEMETRY ---

def chop(ingredient: str) -> str:
    """Chops a raw ingredient. Example: 'tomato' -> 'chopped tomato'"""
    print(f"  [Action] 🔪 Chopping '{ingredient}'...")
    if ingredient not in kitchen.inventory:
        return f"Error: '{ingredient}' is not available in inventory. You must use what is existing."
    new_item = f"chopped {ingredient}"
    kitchen.inventory.add(new_item)
    return f"Success: created {new_item}"

def grill(ingredient: str) -> str:
    """Grills a raw ingredient. Example: 'potato' -> 'grilled patty'"""
    print(f"  [Action] 🍳 Grilling '{ingredient}'...")
    if ingredient not in kitchen.inventory:
        return f"Error: '{ingredient}' is not available in inventory."
    new_item = "grilled patty" if ingredient == "potato" else f"grilled {ingredient}"
    kitchen.inventory.add(new_item)
    return f"Success: created {new_item}"

def toast(ingredient: str) -> str:
    """Toasts a raw ingredient. Example: 'bread' -> 'toasted bun'"""
    print(f"  [Action] 🍞 Toasting '{ingredient}'...")
    if ingredient not in kitchen.inventory:
        return f"Error: '{ingredient}' is not available in inventory."
    new_item = "toasted bun" if ingredient == "bread" else f"toasted {ingredient}"
    kitchen.inventory.add(new_item)
    return f"Success: created {new_item}"

def boil(ingredient: str) -> str:
    """Boils an ingredient. Example: 'pasta' -> 'boiled pasta'"""
    print(f"  [Action] 💧 Boiling '{ingredient}'...")
    if ingredient not in kitchen.inventory:
        return f"Error: '{ingredient}' is not available in inventory."
    new_item = f"boiled {ingredient}"
    kitchen.inventory.add(new_item)
    return f"Success: created {new_item}"

def combine(ingredients: list[str], new_dish_name: str) -> str:
    """Combines multiple existing items from the inventory into a completed dish name."""
    print(f"  [Action] 🥣 Combining {ingredients} into '{new_dish_name}'...")
    missing = kitchen.check_missing(ingredients)
    if missing:
        return f"Error: Cannot combine. Missing items from current inventory: {', '.join(missing)}."
    kitchen.inventory.add(new_dish_name)
    return f"Success: Combined ingredients into {new_dish_name}"

def serve(dish_name: str) -> str:
    """Serves the final completed dish to the customer to finish the order workflow."""
    print(f"  [Action] 🛎️ Serving '{dish_name}'!")
    if dish_name not in kitchen.inventory:
        return f"Error: '{dish_name}' does not exist in inventory yet. You must combine it first."
    kitchen.served_dish = dish_name
    # Force break the agent loop here so it doesn't hang on a trailing turn
    raise OrderServedInterrupt(f"Order successfully fulfilled: {dish_name}")

cooking_tools = [chop, grill, toast, boil, combine, serve]

In [6]:
# --- 4. VERIFICATION STAGE ---
def verify_order(order: str, served_dish: str) -> dict:
    if not served_dish:
        return {"is_match": False, "reason": "No dish was served."}
        
    prompt = f"Customer Order: {order}\nServed Dish: {served_dish}\nDoes this satisfy the order?"
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="You are a food quality inspector. Compare the order with the served dish and determine if it semantically matches.",
            response_mime_type="application/json",
            temperature=0.0,
            response_schema={
                "type": "OBJECT",
                "properties": {
                    "is_match": {"type": "BOOLEAN"},
                    "reason": {"type": "STRING"}
                },
                "required": ["is_match", "reason"]
            }
        )
    )
    return json.loads(response.text)

In [7]:
# --- 5. MAIN WORKFLOW CORE ---
def process_order(order: str):
    print(f"\n{'='*50}\n[NEW ORDER RECEIVED]: {order}\n{'='*50}")
    kitchen.reset()
    
    chat = client.chats.create(
        model=MODEL_ID,
        config=types.GenerateContentConfig(
            tools=cooking_tools,
            temperature=0.0,
            system_instruction=(
                "You are an autonomous AI chef. You must fulfill the customer's order using ONLY items available in the provided inventory list. "
                "Check tool definitions and docstrings to see how base ingredients transform "
                "(e.g., you must grill 'potato' to get a 'grilled patty', and toast 'bread' to get a 'toasted bun'). "
                "Do not try to grill or toast words that aren't in your inventory. "
                "Once all components are created, use the 'combine' tool, then call the 'serve' tool immediately."
            )
        )
    )
    
    print("AI Chef is planning and cooking...\n")
    
    # Construct a message passing the explicit, live current inventory state
    structured_prompt = f"Customer Order: {order}\nInitial Available Kitchen Inventory: {list(kitchen.inventory)}"
    
    try:
        chat.send_message(structured_prompt)
    except OrderServedInterrupt as e:
        print(f"\n[SYSTEM NOTICE]: {e}")
    except Exception as e:
        print(f"\n[SYSTEM ERROR]: An unexpected execution error occurred: {e}")
    
    print(f"\n[FINAL INVENTORY]: {kitchen.inventory}")
    print(f"[SERVED DISH]: {kitchen.served_dish}\n")
    
    print("--- Verification Stage ---")
    verification_result = verify_order(order, kitchen.served_dish)
    
    status = "SUCCESS" if verification_result.get("is_match") else "FAILED"
    print(f"Result: Verification {status}")
    print(f"Reason: {verification_result.get('reason')}")
    print("="*50)

In [8]:
# --- 6. EXECUTE TEST SUITE ---
process_order("Burger")
process_order("Pasta with chopped tomato and cheese")


[NEW ORDER RECEIVED]: Burger
AI Chef is planning and cooking...

  [Action] 🍳 Grilling 'potato'...
  [Action] 🍞 Toasting 'bread'...
  [Action] 🔪 Chopping 'tomato'...
  [Action] 🔪 Chopping 'onion'...
  [Action] 🥣 Combining ['grilled patty', 'toasted bun', 'chopped tomato', 'chopped onion', 'lettuce', 'cheese'] into 'Burger'...
  [Action] 🛎️ Serving 'Burger'!

[FINAL INVENTORY]: {'potato', 'butter', 'pizza sauce', 'pizza dough', 'tomato', 'chopped tomato', 'onion', 'chopped onion', 'bread', 'cheese', 'salt', 'pasta', 'grilled patty', 'toasted bun', 'oil', 'Burger', 'lettuce'}
[SERVED DISH]: Burger

--- Verification Stage ---
Result: Verification SUCCESS
Reason: The served dish matches the customer's order exactly.

[NEW ORDER RECEIVED]: Pasta with chopped tomato and cheese
AI Chef is planning and cooking...

  [Action] 💧 Boiling 'pasta'...
  [Action] 🔪 Chopping 'tomato'...
  [Action] 🥣 Combining ['boiled pasta', 'chopped tomato', 'cheese'] into 'Pasta with chopped tomato and cheese'...
